# Recompute a student's number under the student's own definitions

Use this notebook to grade **Accuracy** for Variations 1 and 3 (Variation 2 has an answer key).

1. Pick three headline numbers from the student's site.
2. For each one, read the student's stated definitions in the site's data notes or README: which sales they kept, how they defined the measure, and any filter on the page (campus, distance, budget).
3. Fill those definitions into `recompute(...)` and compare the result with the number on the site.

**How to read the result.** If the notebook reproduces the student's number, Accuracy is met for that number, even when the definition is a weak one. Judge the definition itself under **Data handling**. If the notebook cannot reproduce it, look for an unstated rule before you mark it wrong: an extra filter, a different date window, a mean instead of a median.

Run the cells from the top. The notebook reads the dataset CSV in `output/students/data/`.

In [1]:
import pandas as pd

from definitions import load

df = load()  # every column, with ZIP, FIPS and parcel numbers kept as text
df["days_to_contract"] = (df["contract_date"] - df["list_date"]).dt.days
df["days_to_close"] = (df["sale_date"] - df["list_date"]).dt.days
print(f"{len(df):,} sales loaded")

35,442 sales loaded


## The recompute function

Every argument is optional. Leave out any rule the student did not state.

| Argument | What it does |
|---|---|
| `measure` | `"sale_price"`, `"price_per_sqft"`, `"days_on_market"` (to `contract_date`), `"days_to_close"` (to `sale_date`), `"year_built"`, or `"count"` |
| `stat` | `"median"` (default) or `"mean"` |
| `min_price`, `max_price` | Price rule, inclusive |
| `min_sqft` | Smallest living area kept for price per square foot |
| `university` | One campus area, as in the `university` column |
| `min_miles`, `max_miles` | Distance rule on `miles_to_campus`; `max_miles` is inclusive unless `max_miles_strict=True` |
| `sold_from`, `sold_to` | Sale date window, inclusive (`"YYYY-MM-DD"`) |
| `listed_from`, `listed_to` | Listing date window, inclusive |
| `budget` | Keep sales at or under this price |

In [2]:
MEASURES = {
    "sale_price": "sale_price",
    "price_per_sqft": "price_per_sqft",
    "days_on_market": "days_to_contract",
    "days_to_close": "days_to_close",
    "year_built": "year_built",
}


def recompute(measure, stat="median", min_price=None, max_price=None, min_sqft=None, university=None,
              min_miles=None, max_miles=None, max_miles_strict=False, sold_from=None, sold_to=None,
              listed_from=None, listed_to=None, budget=None):
    d = df
    if min_price is not None:
        d = d[d["sale_price"] >= min_price]
    if max_price is not None:
        d = d[d["sale_price"] <= max_price]
    if budget is not None:
        d = d[d["sale_price"] <= budget]
    if university is not None:
        d = d[d["university"] == university]
    if min_miles is not None:
        d = d[d["miles_to_campus"] >= min_miles]
    if max_miles is not None:
        d = d[d["miles_to_campus"] < max_miles] if max_miles_strict else d[d["miles_to_campus"] <= max_miles]
    if sold_from is not None:
        d = d[d["sale_date"] >= sold_from]
    if sold_to is not None:
        d = d[d["sale_date"] <= sold_to]
    if listed_from is not None:
        d = d[d["list_date"] >= listed_from]
    if listed_to is not None:
        d = d[d["list_date"] <= listed_to]
    if measure == "count":
        return len(d), len(d)
    if measure == "price_per_sqft":
        area = d["living_area_sqft"]
        keep = area > 0 if min_sqft is None else area >= min_sqft
        values = (d["sale_price"] / area)[keep]
    else:
        values = d[MEASURES[measure]]
    values = values.dropna().astype(float)
    result = values.median() if stat == "median" else values.mean()
    return result, len(values)


def show(label, result):
    value, n = result
    print(f"{label}: {value:,.2f} (from {n:,} sales)")

## Example 1: a price premium near campus (adapted from the Variation 3 pilot)

The site's headline: homes **less than 2 miles** from campus cost **19.6% more per square foot** than homes farther out ($248 against $208).

The student's stated definitions:
- Sales from $10,000 to $5,000,000, inclusive.
- Price per square foot only where living area is at least 200 sq ft.
- Close: `miles_to_campus` under 2. Far: 2 or more.

In [3]:
close = recompute("price_per_sqft", min_price=10_000, max_price=5_000_000, min_sqft=200, max_miles=2, max_miles_strict=True)
far = recompute("price_per_sqft", min_price=10_000, max_price=5_000_000, min_sqft=200, min_miles=2)
show("Close, median price per sq ft", close)
show("Far, median price per sq ft", far)
print(f"Premium: {close[0] / far[0] - 1:.1%}")

Close, median price per sq ft: 248.39 (from 6,683 sales)
Far, median price per sq ft: 207.64 (from 28,187 sales)
Premium: 19.6%


The notebook reproduces $248.39, $207.64 and 19.6%, so Accuracy is met for this number. The $200 sq ft floor and the $10,000 to $5,000,000 price rule are reasonable and stated, so they also satisfy Data handling.

## Example 2: median days on market (adapted from the Variation 1 pilot)

The site's headline: a median of **11 days** on market, from **28,028** sales.

The student's stated definitions:
- Sales from $10,000 to $5,000,000, inclusive.
- Days on market is `contract_date` minus `list_date`, only for sales with both dates.

In [4]:
show("Median days on market", recompute("days_on_market", min_price=10_000, max_price=5_000_000))

Median days on market: 11.00 (from 28,028 sales)


Reproduced: 11 days from 28,028 sales. The count differs from the 28,042 in the data description sheet only because of the student's price rule, which the student explained.

## Example 3: a reproducible number built on a wrong definition

Suppose a student's site says homes around Michigan take **39 days** on market, and the data notes define days on market as `sale_date` minus `list_date`.

In [5]:
show("Michigan, sale_date minus list_date (the student's definition)",
     recompute("days_to_close", university="University of Michigan"))
show("Michigan, contract_date minus list_date (the description sheet's definition)",
     recompute("days_on_market", university="University of Michigan"))

Michigan, sale_date minus list_date (the student's definition): 39.00 (from 2,505 sales)
Michigan, contract_date minus list_date (the description sheet's definition): 7.00 (from 2,450 sales)


The student's 39 days is reproducible under the student's own definition, so **Accuracy** is met. But the definition contradicts the data description sheet, which says days on market ends at `contract_date`; the 39 days includes about a month of closing time. Mark that under **Data handling**, not Accuracy.

## Your student's number

Copy the cell below, fill in the student's rules, and compare.

In [6]:
# show("Student's number", recompute("sale_price", min_price=1_000, university="University of Georgia"))